# Basics &mdash; Quantifiers as Repeated Conjunction and Disjunction

**Concept 8 of the Basics decomposition:** *Quantifiers as Repeated Conjunction and Disjunction, and Negating Them*

$\forall$ is an infinite $\wedge$, $\exists$ an infinite $\vee$ &mdash; and negation swaps them.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Quantifiers/Concept-Quantifiers.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Read the quantifiers as repeated connectives:

$$\forall x \in S:\ p(x) \ \equiv\ p(s_1)\wedge p(s_2)\wedge\cdots \qquad
\exists x \in S:\ p(x) \ \equiv\ p(s_1)\vee p(s_2)\vee\cdots$$

Everything then follows from DeMorgan. **Negation swaps them and pushes inward:**

$$\neg\forall x:\ p(x) \ \equiv\ \exists x:\ \neg p(x) \qquad
\neg\exists x:\ p(x) \ \equiv\ \forall x:\ \neg p(x)$$

Two consequences:

* over an **empty** $S$, $\forall$ is **true** (empty conjunction) and $\exists$ is
  **false** (empty disjunction);
* **order matters**: $\forall x\exists y$ is weaker than $\exists y\forall x$.

This is the machinery Chapter 4 uses to negate the Pumping Lemma's four nested
quantifiers mechanically rather than by feel.

## 2. Definitions

### Quantifiers as folds

In [ ]:
from functools import reduce
def forall(S, p): return reduce(lambda a, b: a and b, (p(x) for x in S), True)
def exists(S, p): return reduce(lambda a, b: a or b,  (p(x) for x in S), False)

### The Pumping Lemma's quantifier stack, as a checkable predicate

In [ ]:
def cond(in_L, Ns, Ws, splits, Is):
    # exists N : forall w : exists split : forall i : ...
    return exists(Ns, lambda N:
             forall(Ws(N), lambda w:
               exists(splits(w, N), lambda sp:
                 forall(Is, lambda i: in_L(sp[0] + sp[1] * i + sp[2])))))

<!-- nav-strip -->

---

&larr;&nbsp;[Basics&nbsp;7.&nbsp;DeMorgan's Law and the Contrapositive Form](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-DeMorgan-And-Contrapositive/Concept-DeMorgan-And-Contrapositive.ipynb) &nbsp;&middot;&nbsp; [**Basics** index](https://github.com/ganeshutah/Jove/blob/master/Basics/README.md) &nbsp;&middot;&nbsp; [Basics&nbsp;9.&nbsp;Negating Implication, and Moving Terms Across It](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Negating-Implication/Concept-Negating-Implication.ipynb)&nbsp;&rarr;

---

## 3. Tests

$\forall$ is an AND-fold; $\exists$ is an OR-fold.

In [ ]:
S = list(range(1, 7))
print("  forall x in 1..6 : x > 0   ->", forall(S, lambda x: x > 0))
print("  forall x in 1..6 : x > 3   ->", forall(S, lambda x: x > 3))
print("  exists x in 1..6 : x > 3   ->", exists(S, lambda x: x > 3))
print("  exists x in 1..6 : x > 9   ->", exists(S, lambda x: x > 9))
assert forall(S, lambda x: x > 0) and not forall(S, lambda x: x > 3)
assert exists(S, lambda x: x > 3) and not exists(S, lambda x: x > 9)

**Over an empty set:** $\forall$ is true, $\exists$ is false.

In [ ]:
print("  forall x in {} : anything ->", forall([], lambda x: False))
print("  exists x in {} : anything ->", exists([], lambda x: True))
assert forall([], lambda x: False) is True
assert exists([], lambda x: True) is False
print("\nEmpty AND-fold is the identity of AND (True);")
print("empty OR-fold is the identity of OR (False).")

**Negation swaps them**, exhaustively over a range of predicates.

In [ ]:
preds = [lambda x: x > 3, lambda x: x % 2 == 0, lambda x: True, lambda x: False]
for i, p in enumerate(preds):
    a = not forall(S, p)
    b = exists(S, lambda x: not p(x))
    c_ = not exists(S, p)
    d = forall(S, lambda x: not p(x))
    print("  pred %d : !forall == exists! -> %-6s   !exists == forall! -> %s"
          % (i, a == b, c_ == d))
    assert a == b and c_ == d

**Order matters:** $\forall x\exists y$ is weaker than $\exists y\forall x$.

In [ ]:
S = [1, 2, 3]
eq = lambda x, y: x == y
ae = forall(S, lambda x: exists(S, lambda y: eq(x, y)))     # y may depend on x
ea = exists(S, lambda y: forall(S, lambda x: eq(x, y)))     # ONE y for every x
print("  forall x exists y : x = y  ->", ae, "   (take y = x)")
print("  exists y forall x : x = y  ->", ea, "   (no single y equals 1, 2 AND 3)")
assert ae is True and ea is False
print()
print("Swapping the two quantifiers turned a TRUE claim into a FALSE one.")
print("The difference is whether the witness y is allowed to depend on x.")
print()
# and the direction of the implication that DOES hold
both = [(lambda x, y: x == y), (lambda x, y: x <= y), (lambda x, y: True)]
for i, r in enumerate(both):
    a = forall(S, lambda x: exists(S, lambda y: r(x, y)))
    b = exists(S, lambda y: forall(S, lambda x: r(x, y)))
    print("  relation %d : forall-exists %-6s exists-forall %-6s   ea => ae ? %s"
          % (i, a, b, (not b) or a))
    assert (not b) or a          # exists-forall is the STRONGER statement
print()
print("exists y forall x  =>  forall x exists y, never the converse.")

The Pumping Lemma's stack, evaluated for a regular language.

In [ ]:
def in_L3Z(s): return s.count('0') % 3 == 0
def splits(w, N):
    return [(w[:i], w[i:j], w[j:]) for i in range(N + 1)
            for j in range(i + 1, min(N, len(w)) + 1)]
def Ws(N):
    from itertools import product
    return [''.join(p) for k in range(N, N + 3)
            for p in product('01', repeat=k) if in_L3Z(''.join(p))]
got = cond(in_L3Z, [3], Ws, splits, range(4))
print("Cond(L3Z) holds (N = 3) ? ", got)
assert got
print("\nFour nested quantifiers, evaluated mechanically.  Chapter 4, Concept 22")
print("negates exactly this stack -- and each flip is one DeMorgan step.")

## 4. Exercises


1. Negate $\forall x \exists y \forall z:\ p(x,y,z)$ mechanically.
2. Give predicates where $\forall x\exists y$ is true but $\exists y\forall x$ is false.
3. Why is "all unicorns are purple" true?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Basics/Concept-Quantifiers')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')